# Business Layer & Reporting
**Mục tiêu:** Tạo các Views phục vụ báo cáo và thuật toán phát hiện bất thường (Alert).
- **Schema:** `business`
- **Tech:** SQL Views, Python for Anomaly Detection.

In [0]:
catalog_name = "brazilian_ecommerce"
gold_schema = "gold"
business_schema = "business"

# Tạo schema Business
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{business_schema}")
print(f"Target: {catalog_name}.{business_schema}")

In [0]:
# Tạo View: Doanh thu theo Tháng (Monthly Sales)
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{business_schema}.vw_sales_monthly_trend AS
SELECT 
    d.year,
    d.month,
    d.month_name,
    d.date_id, -- Dùng để sort biểu đồ cho đúng thứ tự tháng
    COUNT(DISTINCT f.order_id) as total_orders,
    SUM(f.sales_amount) as total_revenue,
    AVG(f.sales_amount) as avg_ticket_size
FROM {catalog_name}.{gold_schema}.fact_sales f
JOIN {catalog_name}.{gold_schema}.dim_date d ON f.date_id = d.date_id
GROUP BY d.year, d.month, d.month_name, d.date_id
""")

print("✅ View `vw_sales_monthly_trend` created.")

In [0]:
# Tạo View: Top Danh mục sản phẩm bán chạy
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{business_schema}.vw_product_performance AS
SELECT 
    p.category_english as category_name,
    COUNT(DISTINCT f.order_id) as orders_count,
    SUM(f.sales_amount) as total_revenue,
    RANK() OVER (ORDER BY SUM(f.sales_amount) DESC) as revenue_rank
FROM {catalog_name}.{gold_schema}.fact_sales f
JOIN {catalog_name}.{gold_schema}.dim_products p ON f.product_sk = p.product_sk
WHERE p.category_english IS NOT NULL
GROUP BY p.category_english
""")

print("✅ View `vw_product_performance` created.")

In [0]:
# Tạo View: Doanh thu theo Bang
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{business_schema}.vw_customer_geography AS
SELECT 
    c.customer_state,
    COUNT(DISTINCT c.customer_natural_key) as active_customers,
    SUM(f.sales_amount) as total_revenue
FROM {catalog_name}.{gold_schema}.fact_sales f
JOIN {catalog_name}.{gold_schema}.dim_customers c ON f.customer_sk = c.customer_sk
WHERE c.is_current = true
GROUP BY c.customer_state
""")

print("✅ View `vw_customer_geography` created.")